In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ L = -y\ln(p)-(1-y)\ln(1-p) $$
$$ \frac{\partial L}{\partial p} = -y\frac{1}{p} + (1-y)\frac{1}{1-p} = \frac{p-y}{p(1-p)} $$

$$ \diamond \diamond \diamond $$

$$ \text{Let } p = sigmoid(z) $$
$$ \frac{\partial L}{\partial z} = \frac{\partial L}{\partial p} \cdot \frac{\partial p}{\partial z} = \frac{p-y}{p(1-p)} \cdot p(1-p) = p-y $$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from approx import approx # type: ignore

def _bce(p: tr.Tensor, t: tr.Tensor, reduction = "mean") -> tr.Tensor:
    """
    Computes the binary cross-entropy loss between predictions `p` and targets `t` with `mean` reduction.
    Note: This function assumes that `p` has already been clamped to avoid log(0) issues.
    """

    L = - (t * p.log() + (1 - t) * (1 - p).log())
    
    if reduction != "mean":
        assert False, f"Unsupported reduction: {reduction}"
        
    L = L.mean()
    return L

def bce(p: tr.Tensor, t: tr.Tensor, reduction = "mean") -> tr.Tensor:
    """
    Computes the binary cross-entropy loss between predictions `p` and targets `t` with `mean` reduction.
    Note: This function clamps `p` to avoid log(0) issues.
    """

    assert p.shape == t.shape
    assert tr.all((t == -1.0) | (t == +1.0))
    assert tr.all((p >= -1.0) & (p <= +1.0))

    eps = 1e-7
    pc = p.clamp(eps, 1 - eps)
    
    return _bce(pc, t, reduction)
    

class BCEFunction(autograd.Function):
    """Custom autograd function for the binary cross-entropy loss."""

    @staticmethod
    def forward(ctx, p: tr.Tensor, t: tr.Tensor) -> tr.Tensor:
        eps = 1e-7
        pc = p.clamp(eps, 1 - eps)
        ctx.save_for_backward(pc, t)
        return _bce(pc, t)

    @staticmethod
    def backward(ctx, grad_output: tr.Tensor) -> tuple[tr.Tensor, None]:        
        (p, t) = ctx.saved_tensors
        S = p.shape[0]
        FO = p.shape[1]
        N = S * FO

        df_dp = (p - t) / (p * (1 - p)) 
        
        # Average over all elements
        df_dp = df_dp / N

        # For this example grad_output == 1
        df_dp = df_dp * grad_output
        assert df_dp.shape == (S, FO)

        return (df_dp, None)
   

class BCE(nn.Module):
    """Custom module for the binary cross-entropy loss."""

    def __init__(self, reduction: str = "mean"):
        super().__init__()

        if reduction != "mean":
            assert False, f"Unsupported reduction: {reduction}"
        
    def forward(self, p: tr.Tensor, t: tr.Tensor) -> tr.Tensor:
        return BCEFunction.apply(p, t)

In [ ]:
# Unit tests

def test_BCEFunction_gradcheck(samples=10):
    def fn(p, y):
        return BCEFunction.apply(p, y)

    p = tr.rand(samples, 1, dtype=tr.float64, requires_grad=True)
    y = tr.randint(0, 2, (samples, 1)).float().requires_grad_(False)
    assert autograd.gradcheck(fn, (p, y), eps=1e-6, atol=1e-4, rtol=1e-4)


def test_BCE_compare(samples):
    p = tr.rand(samples, 1, requires_grad=True)
    y = tr.randint(0, 2, (samples, 1)).float()

    p1 = p.clone().detach().requires_grad_(True)
    y1 = BCE(reduction='mean')(p1, y)
    y1.backward()

    p2 = p.clone().detach().requires_grad_(True)
    y2 = nn.BCELoss(reduction='mean')(p2, y)
    y2.backward()

    assert y1.item() == approx(y2.item())
    assert p1.grad == approx(p2.grad)


if __name__ == "__main__":
    test_BCEFunction_gradcheck(1)
    test_BCEFunction_gradcheck(10)
    test_BCEFunction_gradcheck(100)

    test_BCE_compare(1)
    test_BCE_compare(10)
    test_BCE_compare(100)